In [ ]:
import os, pandas as pd, json
from datetime import datetime
# Install TinyDB to simulate a NoSQL document store locally
!pip install tinydb -q
from tinydb import TinyDB, Query

# --- 1. Data Acquisition (Movies Dataset) ---
# Note: Ensure you have uploaded your kaggle.json to the /content/ folder
if not os.path.exists('movies_metadata.csv'):
    try:
        !kaggle datasets download -d rounakbanik/the-movies-dataset -f movies_metadata.csv
        !unzip -o movies_metadata.csv.zip
    except:
        print('Kaggle API not configured. Please upload kaggle.json to /content/')

# Loading a subset for high-performance processing
df_movies = pd.read_csv('movies_metadata.csv', low_memory=False).head(10000)
print(f'Setup complete. Loaded {len(df_movies)} movie records.')

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
100% 12.2M/12.2M [00:00<00:00, 146MB/s]

Archive:  movies_metadata.csv.zip
  inflating: movies_metadata.csv     
Setup complete. Loaded 10000 movie records.


In [ ]:
# --- 2. Intermediate NoSQL Document Modeling ---
# This is the core skill: Transforming relational strings into nested JSON documents

def clean_json_col(val):
    try:
        # Convert string representations of lists/dicts into actual Python objects
        return json.loads(str(val).replace("'", '"'))
    except:
        return []

def build_movie_documents(dataframe):
    print('Transforming Relational Data to Nested Document Model...')
    docs = []
    for _, row in dataframe.iterrows():
        # Nesting multiple related entities into a single NoSQL document
        doc = {
            'id': row['id'],
            'title': row['title'],
            'genres': [g['name'] for g in clean_json_col(row['genres'])],
            'production_companies': [c['name'] for c in clean_json_col(row['production_companies'])],
            'release_date': str(row['release_date']),
            'revenue': float(row['revenue']) if pd.notnull(row['revenue']) else 0,
            'vote_average': float(row['vote_average']) if pd.notnull(row['vote_average']) else 0,
            'vote_count': float(row['vote_count']) if pd.notnull(row['vote_count']) else 0
        }
        docs.append(doc)
    return docs

movie_documents = build_movie_documents(df_movies)
print(f'Successfully modeled {len(movie_documents)} NoSQL movie documents.')

Transforming Relational Data to Nested Document Model...
Successfully modeled 10000 NoSQL movie documents.


In [ ]:
# --- 3. Database Operations & Intelligence Query ---
# Initialize local NoSQL store (JSON format)
db = TinyDB('movie_vault.json')
db.truncate() # Clear previous runs
db.insert_multiple(movie_documents)

print('\n--- CINE-INTELLIGENCE REPORT (TOP RATED BLOCKBUSTERS) ---')
Movie = Query()
# Complex NoSQL Query: High rating AND high revenue
trending = db.search((Movie.vote_average >= 7.5) & (Movie.revenue > 100000000))

# Transform back to a viewable DataFrame for the user report
results = pd.DataFrame([{
    'Title': m['title'],
    'Rating': m['vote_average'],
    'Revenue': f"${m['revenue']:,.0f}",
    'Genres': ', '.join(m['genres'][:2])
} for m in trending])

if not results.empty:
    display(results.sort_values('Rating', ascending=False).head(10))
else:
    print('No results found for the query criteria.')

print('\nProject Status: Fully Operational (Local NoSQL Engine)')


--- CINE-INTELLIGENCE REPORT (TOP RATED BLOCKBUSTERS) ---


,Title,Rating,Revenue,Genres
16,The Godfather,8.5,"$245,066,411","Drama, Crime"
11,Schindler's List,8.3,"$321,365,567","Drama, History"
46,Life Is Beautiful,8.3,"$229,400,000","Comedy, Drama"
51,Fight Club,8.3,"$100,853,753",Drama
20,One Flew Over the Cuckoo's Nest,8.3,"$108,981,275",Drama
6,Pulp Fiction,8.3,"$213,928,762","Thriller, Crime"
62,Spirited Away,8.3,"$274,925,095","Fantasy, Adventure"
77,Howl's Moving Castle,8.2,"$234,710,455","Fantasy, Animation"
52,Princess Mononoke,8.2,"$159,375,308","Adventure, Fantasy"
21,The Empire Strikes Back,8.2,"$538,400,000","Adventure, Action"



Project Status: Fully Operational (Local NoSQL Engine)
